# Logic Tensor Networks

## Logic Tensor Networks (LTN)

LTN (Serafini & d'Avila Garcez, 2016) encodes First-Order Logic as a loss function over neural network outputs. The key idea:

- **Grounding**: Map logical constants to real vectors, predicates to neural networks
- **Satisfaction**: Each logical axiom becomes a loss term measuring how well the neural network satisfies the axiom
- **Learning**: Minimize the total unsatisfied constraint, equivalent to making the system more consistent with the knowledge base

FOL operators become differentiable:
- ∧ (AND): prod(a, b) or min(a, b)
- ∨ (OR): 1 - prod(1-a, 1-b)
- ¬ (NOT): 1 - a
- ∀x P(x): mean over groundings

In [ ]:
import numpy as np

np.random.seed(42)

# LTN: Logic Tensor Networks
# Encode FOL axioms as differentiable loss functions

# Task: classify animals as birds/mammals with background knowledge
# Axioms: All flying things are birds, All warm-blooded + fur = mammals

# Generate animal features
N = 100
# Features: [has_wings, warm_blooded, has_fur, can_fly, lays_eggs]
birds = np.column_stack([
    np.random.binomial(1, 0.9, N),  # has_wings
    np.random.binomial(1, 0.85, N), # warm_blooded
    np.random.binomial(1, 0.05, N), # has_fur
    np.random.binomial(1, 0.8, N),  # can_fly
    np.random.binomial(1, 0.9, N),  # lays_eggs
])
mammals = np.column_stack([
    np.random.binomial(1, 0.03, N), # has_wings
    np.random.binomial(1, 0.98, N), # warm_blooded
    np.random.binomial(1, 0.92, N), # has_fur
    np.random.binomial(1, 0.05, N), # can_fly
    np.random.binomial(1, 0.02, N), # lays_eggs
])

X = np.vstack([birds, mammals]).astype(float)
y = np.array([1]*N + [0]*N)  # 1=bird, 0=mammal

# Predicate networks: IsBird(x) and IsMammal(x) as logistic classifiers
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

W_bird = np.random.randn(5) * 0.3
W_mamm = np.random.randn(5) * 0.3

# LTN satisfaction levels
def pred_bird(X): return sigmoid(X @ W_bird)
def pred_mamm(X): return sigmoid(X @ W_mamm)

# Axiom 1: forall x. IsBird(x) AND HasWings(x) AND LaysEggs(x)
# for known birds: all three should be high
def axiom_bird_constraints(X_birds):
    p_bird = pred_bird(X_birds)
    has_wings = X_birds[:, 0]  # grounding of HasWings predicate
    lays_eggs = X_birds[:, 4]
    # AND = product t-norm: p_bird * has_wings * lays_eggs
    satisfaction = (p_bird * has_wings * lays_eggs).mean()
    return satisfaction  # higher = more satisfied

# Axiom 2: forall x. IsMammal(x) AND HasFur(x) AND WarmBlooded(x)
def axiom_mammal_constraints(X_mammals):
    p_mamm = pred_mamm(X_mammals)
    has_fur = X_mammals[:, 2]
    warm_bl = X_mammals[:, 1]
    return (p_mamm * has_fur * warm_bl).mean()

# Axiom 3: exclusivity forall x. NOT(IsBird(x) AND IsMammal(x))
def axiom_exclusivity(X):
    pb = pred_bird(X); pm = pred_mamm(X)
    # NOT(pb AND pm) = 1 - pb * pm
    return (1 - pb * pm).mean()

# Training loop: maximize total satisfaction
for _ in range(500):
    # LTN total satisfaction
    s1 = axiom_bird_constraints(X[:N])
    s2 = axiom_mammal_constraints(X[N:])
    s3 = axiom_exclusivity(X)
    total = (s1 + s2 + s3) / 3
    
    # Gradient update (numerical)
    for W_var in [W_bird, W_mamm]:
        for i in range(len(W_var)):
            dw = 0.01
            W_var[i] += dw
            sp = (axiom_bird_constraints(X[:N]) + axiom_mammal_constraints(X[N:]) + axiom_exclusivity(X)) / 3
            W_var[i] -= 2*dw
            sm = (axiom_bird_constraints(X[:N]) + axiom_mammal_constraints(X[N:]) + axiom_exclusivity(X)) / 3
            W_var[i] += dw
            W_var[i] += 0.1 * (sp - sm) / (2*dw)

print('LTN Training Results:')
print(f'  Bird constraint satisfaction: {axiom_bird_constraints(X[:N]):.3f}')
print(f'  Mammal constraint satisfaction: {axiom_mammal_constraints(X[N:]):.3f}')
print(f'  Exclusivity satisfaction: {axiom_exclusivity(X):.3f}')

pb = pred_bird(X); pm = pred_mamm(X)
preds = (pb > pm).astype(int)
print(f'  Classification accuracy: {(preds == y).mean():.3f}')
